# BizPilot AI — Sıfırdan Öğretici Not Defteri

**Kime göre yazıldı:** Bilgisayar Mühendisliği 3. sınıf öğrencisi. Yani Python'u biliyorsun, ama ML / RAG / LLM / agent kavramlarını ve bu projedeki *her dosyanın neden var olduğunu* baştan öğrenmek istiyorsun.

**Amaç:** Bu projenin (BizPilot AI) her parçasını — her klasörü, her `.py` dosyasını, önemli fonksiyonların her satırını — *ne yapıyor* ve *neden böyle yapıyoruz* sorularıyla anlatmak.

    "> Bu defter projenin *aynası* gibidir: kodun kendisi `src/` klasöründe durur, biz burada onu parça parça açıklıyoruz. Bir bölümü okuduktan sonra ilgili gerçek dosyayı açıp yan yana bakmak en iyi öğrenme yöntemidir.\n",

---

## Bu defter nasıl kullanılır?

1. **Sırayla oku.** Bölümler birbirinin üzerine inşa ediliyor: önce kavramlar, sonra veri, sonra modeller, sonra RAG, sonra arayüz, en sonda dağıtım (deployment).
2. **Kod hücrelerini çalıştır.** Çoğu hücre gerçekten çalışır ve projeyle aynı fonksiyonları kullanır. Ağır olanlar (embedding modeli indirme, LLM çağrısı) `# OPSİYONEL` etiketiyle işaretli — internet/paket gerektirir.
3. **Merak ettiğinde gerçek dosyayı aç.** Her bölümün başında hangi dosyayı işlediğimizi yazıyorum.

## İçindekiler

| # | Bölüm | İşlenen dosya(lar) |
|---|-------|--------------------|
| 0 | Bu defter + zihinsel harita | — |
| 1 | Proje ne yapıyor? Mimari ve haftalık plan | `README.md`, `anagorev.md` |
| 2 | Ortam, bağımlılıklar, klasör yapısı | `requirements.txt`, kök klasör |
| 3 | Temel kavramlar (ML, embedding, vektör DB, RAG, agent) | — |
| 4 | Veri katmanı: 3 farklı veri kümesi | `data/...` |
| 5 | Lead Scoring — baseline model (satır satır) | `src/lead_scoring_baseline.py` |
| 6 | Lead Scoring — tahmin + kural katmanı | `src/lead_scoring_predictor.py` |
| 7 | Lead Scoring — toplu skorlama | `src/lead_scoring_batch.py` |
| 8 | Lead Scoring — LLM açıklayıcı | `src/lead_scoring_llm_explainer.py` |
| 9 | Lead Scoring — model karşılaştırma | `src/lead_scoring_model_comparison.py` |
| 10 | RAG pipeline (satır satır) | `src/rag_pipeline.py` |
| 11 | LLM'e gerçek çağrı (OpenAI, stdlib) | `src/lead_scoring_llm_explainer.py` |
| 12 | Agentic Outreach üretici | `src/outreach_agent.py` |
| 13 | Rakip istihbaratı | `src/competitor_intel.py` |
| 14 | RAG değerlendirme (RAGAS mantığı) | `src/rag_eval.py` |
| 15 | Streamlit arayüzü | `app.py` |
| 16 | Dağıtım (sunucu, systemd, nginx, Cloudflare) | — |
| 17 | Özet, öğrenme yol haritası, alıştırmalar | — |

## Bölüm 0 — Zihinsel harita: BizPilot AI aslında ne?

Tek cümleyle: **BizPilot AI, bir şirketin satış/iş geliştirme ekibine yardım eden yapay zekâ destekli bir asistan.** Beş işi bir arada yapıyor:

1. **Doküman soru-cevap (RAG):** Şirket dokümanlarından, kaynak göstererek soru cevaplar. ("Starter planı kaç dolar?")
2. **Lead skorlama (ML):** Gelen bir müşteri adayının (lead) ne kadar "sıcak" olduğunu 0-100 arası puanlar.
3. **Outreach yazımı (agent):** Bir prospect için kişiselleştirilmiş soğuk e-posta + LinkedIn mesajı taslağı yazar.
4. **Rakip istihbaratı:** Bir rakip/konu hakkında halka açık web bilgisini toplayıp özetler.
5. **Değerlendirme:** RAG'in kalitesini (doğruluk, isabet) ölçer.

Bunların hepsi tek bir **Streamlit** arayüzünde (7 sekme) toplanıyor.

### Neden bu 5 iş?
Bu proje bir üniversite stajı için verilen bir kapsamdan (`anagorev.md`) doğuyor. Profesör "agentic RAG chatbot for digital business development" istiyor. Yani hem **RAG** (retrieval-augmented generation), hem **agentic** (çok adımlı, araç kullanan yapay zekâ), hem de klasik **ML** (lead scoring) bir arada gösterilecek.

### En önemli tasarım prensibi: "graceful degradation" (zarif bozulma)
Projenin her yerinde şunu göreceksin: **API anahtarı yoksa veya bir kütüphane kurulu değilse, program çökmez — daha basit bir yönteme düşer.**

- LLM anahtarı yoksa → RAG cevabı yerine metinden alıntı yapan "extractive" cevap.
- Web arama anahtarı yoksa → yerel korpustan arama.
- LangGraph kurulu değilse → adımları düz sırayla çalıştıran basit runner.

Bu, gerçek dünya yazılımında çok değerli bir alışkanlık: **bağımlılıkların yokluğuna dayanıklı olmak.** Kodun her yerinde bu deseni işaret edeceğim.

## Bölüm 1 — Proje ne yapıyor? Mimari ve haftalık plan

> İşlenen dosyalar: `README.md`, `anagorev.md`

### Veri akışı (yüksek seviye)

Aşağıdaki şema, bir kullanıcının mesajının sistemin içinde nasıl dolaştığını gösterir. Bir sonraki hücrede bunu Mermaid ile çizeceğiz; şimdi kelimelerle:

```
Kullanıcı (Streamlit) → Chatbot Intent Router (mesajı sınıflandırır)
                          ├─ RAG Q&A          → ChromaDB'den chunk çek → LLM cevap + kaynak
                          ├─ Lead Qualification → ML modeli + kural katmanı → 0-100 skor
                          ├─ Outreach Agent   → qualify→research→draft→critique → e-posta
                          └─ Competitor Intel → Tavily/SerpAPI/yerel → özet
```

### Profesörün verdiği teknoloji yığını (stack)
`anagorev.md` dosyasında profesör şu araçları öneriyor. Projede hangisinin seçildiğini yanına yazdım:

| Katman | Önerilen | Bu projede seçilen |
|--------|----------|--------------------|
| LLM | Llama/Mistral (Groq/Ollama) veya GPT/Gemini | OpenAI (`gpt-5.4`), anahtar yoksa fallback |
| Orkestrasyon | LangChain + LangGraph/CrewAI | LangGraph (opsiyonel), yoksa sıralı runner |
| Vektör DB | ChromaDB veya FAISS | **ChromaDB** |
| Embedding | all-MiniLM-L6-v2 veya OpenAI | **all-MiniLM-L6-v2** (sentence-transformers) |
| Lead scoring | scikit-learn LogReg veya XGBoost | **LogReg** (üretimde), XGBoost karşılaştırmada |
| Web arama | Tavily veya SerpAPI | İkisi de destekli, yoksa yerel |
| Değerlendirme | RAGAS (faithfulness, precision, relevancy) | Kendi hafif RAGAS-benzeri değerlendirici |
| Arayüz/Dağıtım | Streamlit, HF Spaces/Render | **Streamlit**, sunucuya deploy edildi |

### Haftalık ilerleme (projenin hikâyesi)
Proje `docs/` altında hafta hafta belgelenmiş:

- **Hafta 1:** Proje önerisi, literatür taraması, araç kurulumu, ilk lead-scoring baseline.
- **Hafta 2:** RAG prototipi (yükle → parçala → embed → ChromaDB → sorgu).
- **Hafta 3:** Lead scoring modülünün olgunlaşması (kural katmanı, batch).
- **Hafta 4:** Agentic outreach üretici (qualify→research→draft→critique).
- **Hafta 5:** RAG değerlendirmesi (faithfulness / context precision / answer relevancy + guardrail).
- **Hafta 6:** Dağıtım (deployment) — Streamlit'i canlıya almak.

Bu defterde bu sırayı takip edeceğiz çünkü kod da bu sırayla büyümüş.

In [1]:
# Mimariyi görselleştirelim. (Bu bir Markdown/Mermaid önizlemesi değil, sadece metni yazdırır;
# gerçek diyagramı README.md içindeki mermaid bloğunda VS Code önizlemesiyle görebilirsin.)

architecture = r'''
  [Kullanıcı / Streamlit UI]
            |
     [Intent Router]  <-- app.py: classify_chat_intent()
     /      |      \       \
  RAG    Lead    Outreach  Competitor
   |       |        |          |
 Chroma  LogReg  qualify->   Tavily/
  DB    +kural   research->   SerpAPI/
   |     katmanı draft->      yerel korpus
 LLM     |      critique       |
 cevap   0-100    |          özet
 +kaynak skoru  e-posta+LinkedIn
'''
print(architecture)


  [Kullanıcı / Streamlit UI]
            |
     [Intent Router]  <-- app.py: classify_chat_intent()
     /      |      \       \
  RAG    Lead    Outreach  Competitor
   |       |        |          |
 Chroma  LogReg  qualify->   Tavily/
  DB    +kural   research->   SerpAPI/
   |     katmanı draft->      yerel korpus
 LLM     |      critique       |
 cevap   0-100    |          özet
 +kaynak skoru  e-posta+LinkedIn



## Bölüm 2 — Ortam, bağımlılıklar ve klasör yapısı

> İşlenen dosyalar: `requirements.txt`, kök klasördeki her şey

### Klasör yapısı ve her klasörün görevi

```
BizPilot AI/
├── app.py                  # Streamlit arayüzü — kullanıcının gördüğü her şey (7 sekme)
├── requirements.txt        # Python bağımlılıkları (sabitlenmiş sürümlerle)
├── README.md               # Proje tanıtımı + HF Spaces başlığı + çalıştırma komutları
├── anagorev.md             # Profesörün verdiği kapsam/kurallar (TR çevirisiyle)
├── src/                    # TÜM iş mantığı burada (arayüzden bağımsız, tek başına çalışır)
│   ├── lead_scoring_baseline.py       # Kaggle verisiyle LogReg modeli eğitir
│   ├── lead_scoring_predictor.py      # Tek bir lead'i skorlar (ML + kural)
│   ├── lead_scoring_batch.py          # CSV'deki tüm lead'leri toplu skorlar
│   ├── lead_scoring_llm_explainer.py  # LLM'e çağrı + skoru İngilizce açıklar
│   ├── lead_scoring_model_comparison.py # LogReg/RF/XGBoost karşılaştırır
│   ├── rag_pipeline.py                # RAG: yükle→parçala→embed→index→getir→cevapla
│   ├── outreach_agent.py              # Agentic e-posta üretici
│   ├── competitor_intel.py            # Web/yerel rakip araması + özet
│   └── rag_eval.py                    # RAG kalite değerlendirmesi
├── data/                   # Veri kümeleri (girdi) ve üretilen çıktılar
│   ├── lead_scoring/       # Kaggle ham + temizlenmiş lead verisi
│   ├── crm_sample_leads/   # Örnek CRM lead'leri (batch demo için)
│   └── company_docs/       # RAG korpusu (JSONL) — şirket dokümanları
├── models/                 # Eğitilmiş model dosyası (.joblib)
├── chroma_db/              # ChromaDB'nin diske yazdığı vektör index'i
├── reports/                # Üretilen markdown raporlar (baseline, eval, karşılaştırma)
├── docs/                   # Haftalık ilerleme belgeleri
├── scripts/                # Yardımcı script (synthetic korpus üretici)
└── notebooks/              # Deney/keşif defterleri (bu defter de buraya konabilir)
```

### Önemli mimari kararı: `src/` arayüzden ayrı
`src/` içindeki hiçbir dosya Streamlit'e bağımlı değil. Her biri **komut satırından** (CLI) tek başına çalışabilir. `app.py` sadece bu fonksiyonları çağıran ince bir "kabuk". 

**Neden önemli?** Çünkü iş mantığını arayüzden ayırmak (separation of concerns) test etmeyi, yeniden kullanmayı ve dağıtmayı kolaylaştırır. Örneğin `score_lead()` fonksiyonunu hem `app.py` hem `outreach_agent.py` hem de `lead_scoring_batch.py` kullanıyor.

In [2]:
# Çalıştığımız ortamı ve proje kökünü doğrulayalım.
import sys
from pathlib import Path

# Bu defter notebooks/ altında ya da kökte olabilir; kökü esnek bulalım.
cwd = Path.cwd()
candidates = [cwd, cwd.parent]
ROOT = next((p for p in candidates if (p / 'src' / 'rag_pipeline.py').exists()), cwd)
print('Python sürümü :', sys.version.split()[0])
print('Çalışma dizini:', cwd)
print('Proje kökü    :', ROOT)
print('src/ var mı?  :', (ROOT / 'src').exists())

# src'yi import edilebilir yapalım ki 'from src...' çalışsın.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

Python sürümü : 3.12.10
Çalışma dizini: c:\Users\Semih\Desktop\BizPilot AI\notebooks
Proje kökü    : c:\Users\Semih\Desktop\BizPilot AI
src/ var mı?  : True


### `requirements.txt` — bağımlılıkları neden sabitliyoruz?

Dosya iki gruba ayrılmış:

**1) Çekirdek (zorunlu) — uygulamanın gerçekten `import` ettiği paketler:**
- `streamlit` → web arayüzü
- `pandas`, `numpy` → veri işleme
- `scikit-learn` → lead scoring modeli (LogReg, pipeline, metrikler)
- `joblib` → eğitilmiş modeli diske kaydet/yükle
- `chromadb` → vektör veritabanı
- `sentence-transformers` (+ `torch`, `transformers`, `tokenizers`) → embedding modeli
- `scipy` → sayısal işlemler (bazı bağımlılıklar ister)

**2) Opsiyonel — yorum satırı yapılmış (`#`), sadece ekstra özellik istersen aç:**
- `openai`, `python-dotenv`, `langchain`, `langgraph`, `faiss-cpu`, `ragas`, `tavily-python`, `xgboost`, `google-generativeai` ...

> **Neden sürümler sabitlenmiş (`==`)?** Çünkü "bende çalışıyordu ama sunucuda çöktü" kâbusunu önlemek için. Sabit sürüm = **tekrarlanabilirlik (reproducibility)**. Aynı `requirements.txt` her makinede aynı ortamı kurar.

> **Neden opsiyoneller yorumlanmış?** Çünkü proje bunlar olmadan da çalışır (graceful degradation). Örneğin `openai` paketi bile gerekmiyor — LLM çağrısı Python'un standart `urllib` kütüphanesiyle elle yapılıyor (Bölüm 11). Bu, kurulum yükünü azaltır.

## Bölüm 3 — Temel kavramlar (kod okumadan önce sözlük)

Bu bölümde projede sürekli geçecek terimleri, üniversite dersi netliğinde açıklıyorum. Sonraki bölümlerde bu terimlere geri döneceğiz.

### 3.1 Makine Öğrenmesi (ML) — sınıflandırma
**Lead scoring** bir *ikili sınıflandırma* (binary classification) problemi: bir lead "dönüşür mü (1) / dönüşmez mi (0)". Modeli geçmiş verilerle eğitiyoruz, sonra yeni lead'lerin dönüşme *olasılığını* tahmin ediyoruz.

- **Logistic Regression (Lojistik Regresyon):** Girdileri ağırlıklarla çarpıp toplar, sonucu bir *sigmoid* fonksiyonundan geçirerek 0-1 arası olasılık üretir. Basit, hızlı ve **yorumlanabilir** (hangi özellik skoru artırdı/azalttı görebiliriz). Bu yüzden üretimde bu seçilmiş.
- **ROC-AUC:** Modelin dönüşen/dönüşmeyen lead'leri ne kadar iyi ayırdığını ölçen 0.5-1.0 arası metrik. 0.87 ≈ oldukça iyi.

### 3.2 Embedding (gömme vektörü)
Bir metni (cümle, paragraf) sabit uzunlukta bir *sayı vektörüne* çeviren yapıdır. Anlamca benzer metinler, vektör uzayında birbirine yakın olur. 
- Bu projede `all-MiniLM-L6-v2` modeli her metni **384 boyutlu** bir vektöre çevirir.
- **Normalize edilmiş** vektörlerde iki vektörün *iç çarpımı* = *kosinüs benzerliği* olur (Bölüm 10 ve 14'te kullanacağız).

### 3.3 Vektör veritabanı (ChromaDB)
Embedding'leri saklayan ve "bu sorgu vektörüne en yakın N tanesini getir" diyebildiğimiz özel bir veritabanı. Klasik SQL "eşitlik" arar; vektör DB "yakınlık" arar.

### 3.4 RAG (Retrieval-Augmented Generation)
LLM'ler bilmedikleri (veya uydurdukları) şeyleri söyleyebilir. RAG bunu şöyle çözer:
1. **Retrieve (getir):** Soruyu embed et, vektör DB'den en ilgili doküman parçalarını çek.
2. **Augment (zenginleştir):** Bu parçaları LLM'e "bağlam" olarak ver.
3. **Generate (üret):** LLM sadece bu bağlama dayanarak, kaynak göstererek cevap yazsın.

Böylece cevap **gerçeklere dayalı** ve **kaynaklı** olur. "Hallucination" (uydurma) azalır.

### 3.5 Chunking (parçalama)
Uzun dokümanları LLM'e olduğu gibi veremeyiz (hem uzun hem de retrieval isabetini düşürür). Bu yüzden dokümanları ~900 karakterlik *chunk*'lara böleriz. "Overlap" (örtüşme) ile ardışık chunk'lar biraz ortak metin paylaşır ki bir cümle tam ortadan kesilmesin.

### 3.6 Agent (ajan) ve agentic
Basit LLM çağrısı: soru → cevap. **Agentic** yaklaşım: LLM'i çok adımlı bir *plan* içinde çalıştırır, araçlar kullandırır. Bu projedeki outreach agent 4 adımlıdır: **qualify → research → draft → critique.** Her adım bir "node", hepsi bir "graph" oluşturur.

### 3.7 LLM (Large Language Model)
Metin üreten büyük model (burada OpenAI `gpt-5.4`). Projede LLM'e HTTP üzerinden ham JSON istekle çağrı yapılıyor — hiçbir SDK'ya bağımlı değil (Bölüm 11).

## Bölüm 4 — Veri katmanı: üç farklı veri kümesi

> İşlenen klasör: `data/`

Bu projede birbirinden çok farklı **üç** veri kümesi var. Karıştırmamak önemli çünkü her biri farklı bir modüle hizmet ediyor.

### 4.1 Lead Scoring verisi (ML için) — `data/lead_scoring/`
Bu, halka açık meşhur **Kaggle "Lead Scoring"** veri kümesi. ~9000 satırlık gerçek(imsi) pazarlama verisi.
- `raw/Lead Scoring.csv` → ham hali (indirildiği gibi).
- `processed/lead_scoring_cleaned.csv` → temizlenmiş hali (~7484 satır × 28 sütun). Bunu `lead_scoring_baseline.py` üretir.

Önemli sütunlar:
- `Converted` → **hedef (target)**: 1 = müşteri oldu, 0 = olmadı. Modelin tahmin etmeye çalıştığı şey.
- `Lead Origin`, `Lead Source` → lead nereden geldi (form, Google, referans...).
- `TotalVisits`, `Total Time Spent on Website`, `Page Views Per Visit` → davranış sinyalleri.
- `What is your current occupation` → meslek (çalışan profesyonel dönüşme eğilimi yüksek).
- `Do Not Email`, `Last Activity` → iletişim tercihleri/son etkileşim.

### 4.2 CRM örnek lead'leri (batch demo için) — `data/crm_sample_leads/`
- `crm_leads_sample.csv` → 4 satırlık minik örnek. Gerçek bir CRM'den gelen lead'leri temsil eder (şirket adı, kişi, ünvan, sektör + standart lead alanları).
- `scored_crm_leads.csv` → `lead_scoring_batch.py` bunları skorlayınca oluşan çıktı.

### 4.3 Şirket dokümanları (RAG korpusu) — `data/company_docs/`
- `bizpilot_synthetic_corpus.jsonl` → RAG'in cevap ürettiği bilgi kaynağı. **JSONL** = her satır ayrı bir JSON nesnesi.
- Her kayıt: `doc_id`, `company`, `document_type`, `title`, `source_url`, `retrieved_date`, `corpus_type`, `content`.
- Bu korpus elle yazılmadı; `scripts/generate_synthetic_company_corpus.py` üretti (BizPilot AI'ın ürün/fiyat/SSS/güvenlik dokümanları uydurulmuş ama tutarlı).

> **Neden sentetik korpus?** Çünkü gerçek şirket dokümanı yok (bu bir staj/demo projesi). Ama RAG'i göstermek için tutarlı, kaynak gösterilebilir bir bilgi tabanına ihtiyaç var. Sentetik ama iç tutarlılığı olan bir korpus tam bunu sağlıyor.

Aşağıdaki hücrede bu üç veri kümesini de gerçekten yükleyip ilk satırlarına bakalım.


In [3]:
# Üç veri kümesini de yükleyelim ve tanıyalım.
import json
import pandas as pd

# --- 1) Temizlenmiş lead scoring verisi ---
cleaned_path = ROOT / 'data' / 'lead_scoring' / 'processed' / 'lead_scoring_cleaned.csv'
if cleaned_path.exists():
    leads = pd.read_csv(cleaned_path)
    print('Lead scoring (temiz):', leads.shape, '(satır, sütun)')
    print('Hedef dağılımı (Converted):')
    print(leads['Converted'].value_counts(normalize=True).round(3).to_string())
    display(leads.head(3))
else:
    print('Not: cleaned CSV yok. Önce src/lead_scoring_baseline.py çalıştırılmalı.')

# --- 2) CRM örnek lead'leri ---
crm_path = ROOT / 'data' / 'crm_sample_leads' / 'crm_leads_sample.csv'
if crm_path.exists():
    crm = pd.read_csv(crm_path)
    print('\nCRM örnek lead sayısı:', len(crm))
    display(crm[['crm_lead_id', 'company_name', 'contact_name', 'industry']])

# --- 3) RAG korpusu (JSONL) ---
corpus_path = ROOT / 'data' / 'company_docs' / 'bizpilot_synthetic_corpus.jsonl'
if corpus_path.exists():
    records = [json.loads(line) for line in corpus_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('\nRAG korpus kayıt sayısı:', len(records))
    print('Bir kaydın alanları:', list(records[0].keys()))
    print('Örnek başlık :', records[0]['title'])
    print('İçerik (ilk 200 krk):', records[0]['content'][:200], '...')


Lead scoring (temiz): (7484, 28) (satır, sütun)
Hedef dağılımı (Converted):
Converted
0    0.604
1    0.396


,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Country,...,Newspaper,Digital Advertisement,Through Recommendations,Receive More Updates About Our Courses,Update me on Supply Chain Content,Get updates on DM Content,City,I agree to pay the amount through cheque,A free copy of Mastering The Interview,Last Notable Activity
0,API,Olark Chat,No,No,0,0.0,0,0.0,Page Visited on Website,NaN,...,No,No,No,No,No,No,NaN,No,No,Modified
1,API,Organic Search,No,No,0,5.0,674,2.5,Email Opened,India,...,No,No,No,No,No,No,NaN,No,No,Email Opened
2,Landing Page Submission,Direct Traffic,No,No,1,2.0,1532,2.0,Email Opened,India,...,No,No,No,No,No,No,Mumbai,No,Yes,Email Opened



CRM örnek lead sayısı: 8


,crm_lead_id,company_name,contact_name,industry
0,CRM-001,Northstar CRM Solutions,Aarav Mehta,Software
1,CRM-002,BluePeak Retail,Neha Rao,Ecommerce
2,CRM-003,Campus Labs India,Rohan Sen,Education
3,CRM-004,FinEdge Advisory,Priya Sharma,Financial Services
4,CRM-005,UrbanSupply Co,Arjun Kapoor,Logistics
5,CRM-006,SkillSpring Academy,Isha Verma,Education
6,CRM-007,DataNova Systems,Karan Malhotra,Software
7,CRM-008,GreenCart Foods,Meera Iyer,Food and Beverage



RAG korpus kayıt sayısı: 35
Bir kaydın alanları: ['doc_id', 'company', 'document_type', 'title', 'source_url', 'retrieved_date', 'corpus_type', 'content']
Örnek başlık : BizPilot AI product overview
İçerik (ilk 200 krk): BizPilot AI is an agentic RAG-powered chatbot for digital business development. It combines company-document question answering, lead qualification, personalized outreach generation, competitor-intell ...


## Bölüm 5 — Lead Scoring baseline modeli (satır satır)

> İşlenen dosya: `src/lead_scoring_baseline.py`

Bu dosyanın tek işi var: **ham Kaggle verisini alıp, temizleyip, bir Logistic Regression modeli eğitmek ve diske kaydetmek.** Çıktıları: `models/lead_scoring_logreg.joblib`, `data/lead_scoring/processed/lead_scoring_cleaned.csv` ve `reports/lead_scoring_baseline.md`.

Dosyayı fonksiyon fonksiyon inceleyelim.

### 5.1 Sabitler ve yollar
```python
ROOT_DIR = Path(__file__).resolve().parents[1]
RAW_DATA_PATH = ROOT_DIR / "data" / "lead_scoring" / "raw" / "Lead Scoring.csv"
TARGET_COLUMN = "Converted"
```
- `Path(__file__).resolve().parents[1]` → bu dosya `src/` içinde; `parents[1]` bir üst klasör, yani **proje kökü**. Böylece kod nereden çalıştırılırsa çalıştırılsın yollar doğru kalır. (Mutlak yol yazmaktan kaçınmanın standart yolu.)
- `TARGET_COLUMN = "Converted"` → tahmin edeceğimiz sütun.

```python
DROP_COLUMNS = ["Prospect ID", "Lead Number", "Tags", "Lead Quality", "Lead Profile",
                "Asymmetrique Activity Index", ... ]
```
- **Neden bu sütunlar atılıyor?** İki sebep:
  1. `Prospect ID`, `Lead Number` → sadece kimlik; tahmin gücü yok, ezberlemeye yol açar.
  2. `Tags`, `Lead Quality`, `Asymmetrique...` → bunlar **veri sızıntısı (data leakage)** riski. Muhtemelen lead değerlendirildikten *sonra* elle girilmiş etiketler veya eski skor çıktıları. Modeli bunlarla eğitirsek, gerçek hayatta olmayan bir bilgiyle "hile" yapmış oluruz ve skor yanıltıcı derecede yüksek çıkar.

> **Data leakage** ML'de en sinsi hatadır: eğitimde kullandığın bir sütun, aslında hedefin bir sonucuysa, model kâğıt üzerinde mükemmel görünür ama üretimde işe yaramaz.

### 5.2 `load_dataset()` — CSV'yi oku
Dosya yoksa net bir `FileNotFoundError` fırlatır (sessizce boş dönmez). Aksi halde `pd.read_csv` ile okur.

### 5.3 `clean_dataset(df)` — temizlik
```python
df = df.replace("Select", pd.NA)          # "Select" = kullanıcı seçim yapmamış → eksik değer
df = df.drop(columns=existing_drop_columns) # yukarıdaki sızıntı/ID sütunlarını at
df = df.drop_duplicates()                   # tekrar eden satırları at
df = df.dropna(subset=[TARGET_COLUMN])      # hedefi boş olan satır işe yaramaz → at
```
- `.copy()` ile başlar ki orijinal DataFrame değişmesin (yan etki önleme).
- `existing_drop_columns` → yalnızca *var olan* sütunları atmayı dener (`KeyError` olmasın diye savunmacı programlama).

### 5.4 `build_pipeline(X)` — modelin kalbi
Burada **sayısal** ve **kategorik** sütunlar farklı işlenir:
```python
numeric_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(exclude=["number"]).columns
```
**Sayısal boru hattı:** eksikleri *medyan* ile doldur → `StandardScaler` ile ölçekle (ortalama 0, std 1). LogReg ölçeğe duyarlı olduğu için scaling önemli.

**Kategorik boru hattı:** eksikleri *en sık değer* ile doldur → `OneHotEncoder`:
- `handle_unknown="ignore"` → üretimde eğitimde görülmeyen bir kategori gelirse çökme, sıfır vektör ver.
- `min_frequency=0.01` → %1'den az görülen nadir kategorileri tek bir "diğer" grubunda topla (aşırı sütun patlamasını önler).

**`ColumnTransformer`** bu iki boru hattını doğru sütunlara uygular ve birleştirir.

**Model:**
```python
LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear", random_state=42)
```
- `class_weight="balanced"` → **çok önemli.** Dönüşen lead'ler azınlıkta (dengesiz veri). Bu ayar azınlık sınıfına daha çok ağırlık verir, yoksa model "hep 0 de" diyerek tembellik eder.
- `solver="liblinear"` → küçük/orta veri ve ikili sınıflandırma için iyi çalışan optimizasyon algoritması.
- `random_state=42` → tekrarlanabilirlik (aynı sonuç her seferinde).

Sonunda `preprocessor` + `model` tek bir `Pipeline`'a sarılır. Böylece ham veri girer, tahmin çıkar — ön işleme ve model birlikte kaydedilir/yüklenir.

### 5.5 `get_top_features(pipeline)` — yorumlanabilirlik
LogReg'in her özelliğe verdiği **katsayıyı (coefficient)** çeker. Pozitif katsayı = dönüşümü artırır, negatif = azaltır. En büyük/küçük 10'ar tanesini döner. Bu, "model neye bakıyor?" sorusunun cevabı — LogReg seçilmesinin ana sebeplerinden biri.

### 5.6 `main()` — hepsini birleştirir
1. Klasörleri oluştur.
2. Ham veriyi yükle → temizle.
3. `X` (özellikler) ve `y` (hedef, int'e çevrilmiş) ayır.
4. **`train_test_split(... test_size=0.2, stratify=y, random_state=42)`** → verinin %80'i eğitim, %20'si test. `stratify=y` → hedef oranını iki sette de aynı tutar (dengesiz veride şart).
5. `pipeline.fit(X_train, y_train)` → modeli eğit.
6. `predict` (0/1 sınıf) ve `predict_proba[:,1]` (dönüşme olasılığı) al.
7. Metrikleri hesapla: accuracy, precision, recall, f1, **roc_auc**.
8. Temiz veriyi CSV'ye, modeli `.joblib`'e kaydet, raporu yaz.

**Sonuç:** ROC-AUC ≈ **0.8703**. Yani model dönüşen/dönüşmeyen lead'leri ayırmada oldukça başarılı.


In [4]:
# Projenin GERÇEK build_pipeline fonksiyonunu import edip yapısına bakalım.
# (Modeli yeniden eğitmiyoruz; sadece pipeline'ın nasıl kurulduğunu görüyoruz.)
from src.lead_scoring_baseline import build_pipeline, clean_dataset, TARGET_COLUMN

# Elimizdeki temiz veriden küçük bir X örneği oluşturup pipeline'ı kuralım.
sample_X = leads.drop(columns=[TARGET_COLUMN]).head(50)
pipe = build_pipeline(sample_X)
print('Pipeline adımları:')
for name, step in pipe.named_steps.items():
    print(' -', name, '->', type(step).__name__)

print('\nColumnTransformer içindeki dönüşümler:')
pre = pipe.named_steps['preprocessor']
for name, transformer, cols in pre.transformers:
    print(f'  {name:4s}: {type(transformer).__name__:18s} -> {len(cols)} sütun')


Pipeline adımları:
 - preprocessor -> ColumnTransformer
 - model -> LogisticRegression

ColumnTransformer içindeki dönüşümler:
  num : Pipeline           -> 3 sütun
  cat : Pipeline           -> 24 sütun


## Bölüm 6 — Tek lead skorlama: ML + kural katmanı (satır satır)

> İşlenen dosya: `src/lead_scoring_predictor.py`

Bölüm 5 modeli *eğitti*. Bu dosya ise onu *kullanır*: tek bir lead sözlüğü (`dict`) alır, 0-100 arası skor + etiket + açıklama üretir. En güzel fikir: skoru **sadece ML'e bırakmıyor**, üstüne insan mantığıyla yazılmış **kurallar** ekliyor.

### 6.1 İlginç import bloğu
```python
try:
    from src.lead_scoring_llm_explainer import generate_lead_score_explanation
except ImportError:
    from lead_scoring_llm_explainer import generate_lead_score_explanation
```
- Bu **çift import** deseni her `src/` dosyasında var. Neden? Çünkü kod iki farklı şekilde çalıştırılabiliyor:
  - `app.py` içinden → `from src.xxx` (paket olarak).
  - Doğrudan `python src/lead_scoring_predictor.py` → `from xxx` (aynı klasörden).
- `try/except ImportError` ikisini de destekler. Taşınabilirlik için pratik bir hile.

### 6.2 `load_model()` ve `get_expected_columns(model)`
- `load_model` → `.joblib` dosyasını yükler (yoksa net hata). `joblib`, scikit-learn modellerini (numpy dizileri içerdiği için) `pickle`'dan daha verimli kaydeder.
- `get_expected_columns` → pipeline'ın `preprocessor.feature_names_in_` özelliğinden **modelin eğitimde gördüğü sütun isimlerini** çeker. Bu kritik: tahmin verisi *tam olarak* aynı sütunları içermeli.

### 6.3 `build_input_frame(lead_data, model)` — girdiyi hizala
```python
for column in expected_columns:
    value = lead_data.get(column, np.nan)   # eksik sütun -> NaN
    if isinstance(value, str) and value == "Select":
        value = np.nan
    row[column] = value
frame = pd.DataFrame([row], columns=expected_columns)
```
- Kullanıcı belki 5 alan girdi ama model 27 sütun bekliyor. Bu fonksiyon eksikleri `NaN` ile doldurarak **tek satırlık** bir DataFrame'i modelin beklediği sıraya/şekle sokar. Pipeline'daki imputer bu `NaN`'leri sonra dolduracak.
- Sayısal sütunlar `pd.to_numeric(errors="coerce")` ile güvenle sayıya çevrilir (çevrilemezse `NaN`).

### 6.4 `apply_rule_adjustments(lead_data)` — insan sezgisi katmanı
Bu, **el yapımı kurallar** bloğu. ML skoruna eklenecek/çıkarılacak puan ve gerekçe listesi döner:

| Koşul | Puan | Mantık |
|-------|------|--------|
| `Do Not Email = No` | +4 | E-posta izni var, ulaşabiliriz |
| `Do Not Email = Yes` | −10 | Ulaşamayız, değer düşer |
| `Lead Origin = Lead Add Form` | +8 | Form doldurmak = yüksek niyet |
| `occupation = Working Professional` | +6 | Bütçesi/yetkisi olan profil |
| `occupation = Student/Unemployed` | −4 | Dönüşüm düşük grup |
| Sitede süre ≥ 900 sn | +7 | Yüksek ilgi |
| Sitede süre ≤ 60 sn | −6 | Düşük ilgi |
| Ziyaret ≥ 5 | +4 | Tekrar gelen ilgili |
| Sayfa görüntüleme ≥ 3 | +3 | Aktif gezinme |

> **Neden ML yetmiyor da kural ekliyoruz?** İki sebep: (1) **Yorumlanabilirlik** — satışçıya "neden yüksek?" diye somut gerekçe sunar. (2) **İş bilgisi enjeksiyonu** — "Do Not Email = Yes ise değeri düşür" gibi kurallar veriden öğrenilmese bile iş açısından doğrudur. Bu, ML + kural hibrit yaklaşımı gerçek sistemlerde çok yaygındır.

### 6.5 `score_lead(...)` — hepsini birleştiren ana fonksiyon
```python
ml_probability = float(model.predict_proba(input_frame)[0][1])  # dönüşme olasılığı 0-1
ml_score = round(ml_probability * 100)                           # 0-100
rule_adjustment, reasons = apply_rule_adjustments(lead_data)     # kural puanı
final_score = max(0, min(100, ml_score + rule_adjustment))       # 0-100 arasına sıkıştır
```
- `predict_proba(...)[0][1]` → tek satır (`[0]`), pozitif sınıf olasılığı (`[1]`).
- `max(0, min(100, ...))` → **clamp**: skor asla 0'ın altına / 100'ün üstüne çıkamaz.
- Sonuç sözlüğü: `ml_score`, `rule_adjustment`, `final_score`, `label`, `explanation`, `rule_reasons`.
- `use_llm_explanation=True` ise şablon açıklamayı LLM açıklamasıyla değiştirir (Bölüm 8/11).

### 6.6 `classify_score(score)` — etiketleme
`≥75 → "Yuksek Potansiyel"`, `≥50 → "Orta Potansiyel"`, aksi → `"Dusuk Potansiyel"`. Satışçının hızlı önceliklendirmesi için.

Aşağıda gerçek `score_lead` fonksiyonunu bir örnek lead ile çalıştırıyoruz.


In [5]:
# GERÇEK score_lead fonksiyonunu iki farklı lead ile deneyelim.
# (LLM açıklaması KAPALI; sadece ML + kural katmanı çalışır, API anahtarı gerektirmez.)
from src.lead_scoring_predictor import score_lead, apply_rule_adjustments

sicak_lead = {
    "Lead Origin": "Lead Add Form", "Lead Source": "Google", "Do Not Email": "No",
    "TotalVisits": 6, "Total Time Spent on Website": 1250, "Page Views Per Visit": 3.5,
    "Last Activity": "SMS Sent", "What is your current occupation": "Working Professional",
}
soguk_lead = {
    "Lead Origin": "Landing Page Submission", "Lead Source": "Direct Traffic", "Do Not Email": "Yes",
    "TotalVisits": 1, "Total Time Spent on Website": 30, "Page Views Per Visit": 1.0,
    "Last Activity": "Page Visited on Website", "What is your current occupation": "Student",
}

for isim, lead in [("SICAK", sicak_lead), ("SOGUK", soguk_lead)]:
    r = score_lead(lead, use_llm_explanation=False)
    print(f"=== {isim} LEAD ===")
    print(f"  ML skoru        : {r['ml_score']}/100")
    print(f"  Kural düzeltmesi : {r['rule_adjustment']:+d}")
    print(f"  FINAL skor       : {r['final_score']}/100  -> {r['label']}")
    print(f"  Gerekçeler       : {r['rule_reasons'][:2]}")
    print()


=== SICAK LEAD ===
  ML skoru        : 100/100
  Kural düzeltmesi : +32
  FINAL skor       : 100/100  -> Yuksek Potansiyel
  Gerekçeler       : ['Email izni var, bu outreach icin olumlu bir sinyal.', 'Lead form doldurarak gelmis, bu daha yuksek niyet gosterebilir.']

=== SOGUK LEAD ===
  ML skoru        : 1/100
  Kural düzeltmesi : -23
  FINAL skor       : 0/100  -> Dusuk Potansiyel
  Gerekçeler       : ['Email izni yok, bu satis iletisimini zorlastirabilir.', 'Meslek bilgisi donusum olasiligini biraz dusuren grupta.']



## Bölüm 7 — Toplu (batch) skorlama

> İşlenen dosya: `src/lead_scoring_batch.py`

Bölüm 6 tek lead'i skorluyordu. Gerçek hayatta CRM'de yüzlerce lead vardır. Bu dosya bir CSV'deki **tüm** lead'leri tek tek `score_lead`'den geçirir ve skorlanmış yeni bir CSV yazar.

### Akış — `score_crm_dataset(input_path, output_path, use_llm)`
```python
model = load_model()                 # Modeli BİR KEZ yükle (döngüde tekrar yükleme = israf)
df = pd.read_csv(input_path)
for index, row in df.iterrows():
    lead_data = row.to_dict()
    result = score_lead(lead_data, model=model, use_llm_explanation=use_llm)
    scored_rows.append({**lead_data, "final_score": result["final_score"], ...})
    if batch_delay_seconds > 0 and index < len(df) - 1:
        time.sleep(batch_delay_seconds)   # LLM açıksa istekleri ara-la (rate limit koruması)
output_df.to_csv(output_path, index=False)
```
Önemli detaylar:
- **Model tek sefer yüklenir**, döngüde `model=model` olarak geçilir. Aksi halde her satırda diskten model okumak çok yavaş olurdu.
- `{**lead_data, ...}` → orijinal lead alanları + skor sonuçları tek satırda birleşir. Çıktı hem girdiyi hem skoru içerir (izlenebilirlik).
- **`time.sleep(batch_delay_seconds)`** → yalnızca LLM açıkken. OpenAI'ın *rate limit*'ine (dakikada istek sınırı) takılmamak için istekler arasına gecikme koyar. `get_llm_batch_delay_seconds()` bunu `.env`'den okur, 0-30 sn arasına sıkıştırır.
- `argparse` ile CLI: `python src/lead_scoring_batch.py --input ... --output ... --llm`.

Çıktı: `data/crm_sample_leads/scored_crm_leads.csv`. Aşağıda gerçekten çalıştırıyoruz (LLM kapalı).


In [6]:
# GERÇEK batch skorlamayı CRM örnek verisinde çalıştır (LLM kapalı).
from src.lead_scoring_batch import score_crm_dataset, DEFAULT_INPUT_PATH, DEFAULT_OUTPUT_PATH

scored = score_crm_dataset(DEFAULT_INPUT_PATH, DEFAULT_OUTPUT_PATH, use_llm=False)
print('Skorlanan lead sayısı:', len(scored))
print('Ortalama final skor  :', round(scored['final_score'].mean(), 1))
print('Yüksek öncelikli (>=75):', int((scored['final_score'] >= 75).sum()))
display(scored[['company_name', 'contact_name', 'ml_score', 'rule_adjustment', 'final_score', 'label']])


Skorlanan lead sayısı: 8
Ortalama final skor  : 73.2
Yüksek öncelikli (>=75): 5


,company_name,contact_name,ml_score,rule_adjustment,final_score,label
0,Northstar CRM Solutions,Aarav Mehta,100,32,100,Yuksek Potansiyel
1,BluePeak Retail,Neha Rao,87,10,97,Yuksek Potansiyel
2,Campus Labs India,Rohan Sen,5,-14,0,Dusuk Potansiyel
3,FinEdge Advisory,Priya Sharma,100,32,100,Yuksek Potansiyel
4,UrbanSupply Co,Arjun Kapoor,92,17,100,Yuksek Potansiyel
5,SkillSpring Academy,Isha Verma,44,-9,35,Dusuk Potansiyel
6,DataNova Systems,Karan Malhotra,100,32,100,Yuksek Potansiyel
7,GreenCart Foods,Meera Iyer,58,-4,54,Orta Potansiyel


## Bölüm 8 — LLM açıklayıcı: skoru insana anlat

> İşlenen dosya: `src/lead_scoring_llm_explainer.py` (LLM tarafını Bölüm 11'de detaylandıracağız)

Şu ana kadar skor bir sayı ve birkaç kural gerekçesiydi. Bu dosya, o skoru bir LLM'e verip **akıcı, satış odaklı bir paragrafa** dönüştürür. Ama en önemli özelliği: **API anahtarı yoksa şablon (template) açıklamasına düşer** — çökme yok.

### `generate_lead_score_explanation(lead_data, score_result)`
```python
load_env_file()                                   # .env'i oku
api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key:                                   # anahtar yoksa...
    return {"explanation": build_template_explanation(...),
            "explanation_provider": "template_fallback", ...}   # ...şablona düş
try:
    explanation = call_openai_responses(...)      # anahtar varsa LLM'e sor
    return {"explanation": explanation, "explanation_provider": "openai", ...}
except RuntimeError as exc:
    return {"explanation": build_template_explanation(...), "llm_error": str(exc), ...}
```
Üç senaryo, üç yol — hepsi geçerli bir sonuç döner:
1. Anahtar yok → şablon açıklama.
2. Anahtar var, çağrı başarılı → LLM açıklaması.
3. Anahtar var, çağrı başarısız → yine şablon açıklama + hata mesajı.

Bu, **graceful degradation** deseninin ders kitabı örneği.

### `build_prompt(...)` — LLM'e ne soruyoruz?
Lead profilini ve skor dökümünü **JSON** olarak paketleyip prompt'a gömer. Sistem talimatı çok net kurallar içerir:
- "Sadece verilen JSON'u kullan, eksik CRM verisi uydurma." → **hallucination önleme.**
- "Sadece nihai açıklamayı döndür; analizini/gizli akıl yürütmeni gösterme." → temiz çıktı.
- "3-4 cümlelik tek paragraf; ML skoru, kural düzeltmesi, öncelik ve önerilen aksiyonu belirt." → tutarlı format.

> **Prompt engineering dersi:** İyi bir prompt = rol + kaynak + kısıtlar + format. Bu prompt dördünü de içeriyor.

### `build_template_explanation(...)` — LLM'siz yedek
LLM olmadan, önceden yazılmış İngilizce cümle kalıplarıyla aynı bilgiyi üretir. `LABEL_TRANSLATIONS` ve `REASON_TRANSLATIONS` sözlükleri Türkçe kural gerekçelerini İngilizce'ye çevirir. `next_action` skora göre belirlenir (≥75 → hemen ulaş, <50 → düşük öncelikte tut).

> Bu iki fonksiyonun **aynı bilgiyi** ürettiğine dikkat et: biri LLM'le esnek dille, diğeri sabit şablonla. Kullanıcı hangi modda olduğunu fark etmez bile.


In [8]:
# Şablon (LLM'siz) açıklamayı gerçekten üretelim - anahtar gerektirmez.
from src.lead_scoring_llm_explainer import build_template_explanation

r = score_lead(sicak_lead, use_llm_explanation=False)
print('ŞABLON AÇIKLAMA (LLM olmadan):')
print()
print(build_template_explanation(sicak_lead, r))

# NOT: LLM açıklaması istersen .env içine OPENAI_API_KEY koy ve şunu çalıştır:
#   r2 = score_lead(sicak_lead, use_llm_explanation=True)
#   print(r2['explanation'], '| provider:', r2['explanation_provider'])


ŞABLON AÇIKLAMA (LLM olmadan):

The ML model produced a base score of 100/100. The rule-based layer increased the score by 32 points, resulting in a final score of 100/100 (High Potential). Email outreach is allowed, which is a positive sales signal. The lead came through a form submission, which suggests stronger intent. The working professional profile is a positive conversion signal. Recommended action: prioritize immediate outreach.


## Bölüm 9 — Model karşılaştırma: LogReg mi, XGBoost mu?

> İşlenen dosya: `src/lead_scoring_model_comparison.py`

Baseline'da LogReg seçtik. Ama "daha iyi bir model var mı?" diye sormak bilimsel dürüstlük gerektirir. Bu dosya **aynı veri, aynı bölme, aynı ön işleme** ile üç modeli yarıştırır:

- **Logistic Regression** — basit, hızlı, yorumlanabilir.
- **Random Forest** (`n_estimators=300`) — çok sayıda karar ağacının oylaması; doğrusal olmayan ilişkileri yakalar.
- **XGBoost** (`n_estimators=400, max_depth=5, lr=0.05`) — gradyan artırma; genelde tablosal veride şampiyon. `_HAS_XGBOOST` ile **opsiyonel**: paket yoksa bu model atlanır (yine graceful degradation).

### Adil karşılaştırmanın kuralı
`build_preprocessor` ile **aynı** ColumnTransformer, `evaluate_models` ile **aynı** train/test split üç modele de uygulanır. Böylece fark modelin kendisinden gelir, ön işlemeden değil. Rapor `reports/lead_scoring_model_comparison.md`'ye yazılır.

### Sonuçlar (ROC-AUC)
| Model | ROC-AUC |
|-------|---------|
| Logistic Regression | 0.8703 |
| Random Forest | 0.8670 |
| **XGBoost** | **0.8857** (en iyi) |

### O zaman neden üretimde hâlâ LogReg?
Çünkü fark küçük (~0.015) ama LogReg **yorumlanabilir** (katsayılar = "neden bu skor?") ve **bağımlılık yükü yok** (XGBoost ekstra paket). Staj/demo bağlamında "biraz daha yüksek AUC" yerine "açıklanabilir + hafif" tercih edilmiş. Bu, gerçek mühendislikte sık verilen bir **doğruluk vs. sadelik/yorumlanabilirlik** takasıdır (trade-off).

> Ders: En yüksek skorlu model her zaman "en iyi seçim" değildir. Bağlam (yorumlanabilirlik, bakım maliyeti, gecikme, bağımlılık) kararı belirler.


## Bölüm 10 — RAG pipeline (satır satır) — projenin kalbi

> İşlenen dosya: `src/rag_pipeline.py`

Bu, projenin en önemli dosyası. Bir soruyu alıp şirket dokümanlarından **kaynaklı** cevap üretir. Beş aşama: **yükle → parçala → embed & indexle → getir (hybrid) → cevapla.** Sırayla gidelim.

### 10.1 Sabitler
```python
COMPANY_DOCS_DATASET = .../bizpilot_synthetic_corpus.jsonl  # bilgi kaynağı
CHROMA_DIR = ROOT_DIR / "chroma_db"                         # vektör index diski
COLLECTION_NAME = "bizpilot_company_docs"                   # ChromaDB koleksiyon adı
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DEFAULT_MAX_DISTANCE = 1.45                                 # bundan uzak chunk'lar alakasız sayılır
IDENTITY_QUESTION_PATTERNS = {"what is bizpilot", ...}      # "bu şirket ne?" tipi sorular
```

### 10.2 `DocumentChunk` (dataclass)
```python
@dataclass(frozen=True)
class DocumentChunk:
    chunk_id: str
    text: str
    metadata: dict
```
- `@dataclass` → otomatik `__init__` üretir (az kod). `frozen=True` → değiştirilemez (immutable), yani güvenli. Her chunk'ın kimliği, metni ve meta verisi (kaynak, başlık, tip...) var.

### 10.3 `load_company_documents()` — YÜKLE
JSONL'i satır satır okur. Her satır bir JSON. `require_record_value` ile `doc_id`, `title`, `content` zorunlu alanların dolu olduğunu doğrular (yoksa net hata). `text = title + "\n\n" + content` olarak birleştirir — başlığı da embed'e katmak retrieval'ı güçlendirir. Meta veriyi (kaynak URL, tip, tarih...) saklar; bunlar sonra **alıntı (citation)** için lazım.

### 10.4 `chunk_text(text, max_chars=900, overlap_chars=0)` — PARÇALA
- Metni önce **paragraflara** böler (`re.split(r"\n\s*\n", ...)`). Paragraf sınırları anlamsal olarak doğal kesim noktalarıdır.
- Paragrafları ~900 karaktere kadar biriktirir; sınır aşılınca yeni chunk başlatır. Tek paragraf 900'den uzunsa karakterden keser.
- **`overlap_chars`**: `build_chunks` bunu **150** ile çağırır. Örtüşme, ardışık chunk'ların son 150 karakterini paylaşmasını sağlar ki bir cümle/bilgi tam sınırda kaybolmasın. (Retrieval'da "bağlam kopması" sorununu azaltır.)

### 10.5 `build_chunks(documents)` — chunk nesnelerini üret
Her doküman için `chunk_text(..., overlap_chars=150)` çağırır, her parçaya `doc_id::chunk_003` gibi benzersiz ID + meta veri verir. Bu proje toplam **46 chunk** üretiyor.

### 10.6 `load_embedding_model()` — embed modelini yükle
```python
@lru_cache(maxsize=1)
def load_embedding_model():
    try:
        return SentenceTransformer(EMBEDDING_MODEL_NAME, local_files_only=True)  # önce yereli dene
    except Exception:
        return SentenceTransformer(EMBEDDING_MODEL_NAME)                          # yoksa indir
```
- `@lru_cache(maxsize=1)` → model **bir kez** yüklenir, sonra bellekte tutulur (yüklemek pahalı, tekrarı israf).
- `local_files_only=True` önce → internet yoksa bile çalışsın (offline dayanıklılık). Başarısızsa indirir.

### 10.7 `get_chroma_collection(reset)` ve `build_index()` — EMBED & INDEXLE
```python
client = chromadb.PersistentClient(path=str(CHROMA_DIR))   # diske kalıcı yazar
...
embeddings = model.encode([c.text for c in chunks], normalize_embeddings=True).tolist()
collection.upsert(ids=..., documents=..., embeddings=..., metadatas=...)
```
- **`PersistentClient`** → index diske yazılır (`chroma_db/`), program kapansa bile kalır. Her açılışta yeniden embed etmeye gerek yok.
- **`normalize_embeddings=True`** → çok kritik. Vektörleri birim uzunluğa getirir, böylece *mesafe* ile *kosinüs benzerliği* tutarlı olur (bkz. 10.8).
- **`upsert`** → varsa güncelle, yoksa ekle (insert + update). Tekrar çalıştırınca çift kayıt olmaz.

### 10.8 `retrieve(question, top_k=5, ...)` — GETİR (hybrid re-ranking)
Bu fonksiyonun zekâsı burada. Saf vektör aramasının fiyat/plan adı gibi **anahtar kelime ağırlıklı** sorularda zayıf kaldığı bilinir. Çözüm: **vektör + sözcüksel (lexical) hibrit skorlama.**
```python
query_embedding = model.encode([retrieval_query], normalize_embeddings=True)[0]
fetch_k = top_k * fetch_multiplier          # 5 yerine 20 aday çek (over-fetch)
result = collection.query(query_embeddings=[...], n_results=fetch_k, include=[...distances])
for ... in candidates:
    if distance > max_distance: continue                     # çok uzak = alakasız, ele
    vector_similarity = 1.0 - (distance / 2.0)               # normalize -> mesafe [0,2] -> benzerlik [0,1]
    lexical = _lexical_overlap(retrieval_query, document)    # ortak kelime oranı (Jaccard)
    hybrid_score = 0.75 * vector_similarity + 0.25 * lexical # ağırlıklı birleşim
candidates.sort(by hybrid_score, desc)
return candidates[:top_k]
```
- **Over-fetch (fetch_multiplier=4):** Önce 20 aday çeker, sonra 5'e indirir. Böylece re-ranking'e daha geniş bir havuz sunulur.
- **`_lexical_overlap`:** Sorgu kelimeleri chunk'ta ne kadar geçiyor (basit Jaccard). "Starter" gibi tam kelime eşleşmelerini yakalar — embedding'in gözden kaçırabileceği şey.
- **`build_retrieval_query`:** "What is this company?" gibi çok kısa/kimlik sorularını, arama sinyalini artırmak için zengin bir anahtar kelime dizisine genişletir.

### 10.9 `answer_question(...)` — CEVAPLA (LLM veya extractive)
1. `retrieve` ile chunk'ları çek. Hiç yoksa nazik bir "bulunamadı" mesajı.
2. **Önce her zaman** `generate_extractive_answer` (LLM'siz, chunk'lardan en alakalı cümleleri seçer) hazırlanır — güvenli taban.
3. `use_llm=True` ve LLM çalışıyorsa → `generate_llm_rag_answer` cevabı üstüne yazılır.
4. Çıktıya **"Answer mode"**, **"Relevant context"** (kaynak snippet'leri) ve **"Citations"** (kaynak listesi) eklenir.

### 10.10 `generate_llm_rag_answer(...)` — LLM ile kaynaklı cevap
Sistem prompt'u çok sıkı: *"Sadece getirilen bağlamdan cevapla. Bağlam yetersizse 'dokümanlarda yok' de. [1] [2] gibi alıntı işaretleri kullan. Fiyat/özellik/kaynak uydurma."* Bu, RAG'in **anti-hallucination** güvencesi. `build_rag_generation_prompt` her chunk'ı numaralı bağlam bloğu olarak dizer; LLM bu numaralarla alıntı yapar.

### 10.11 `generate_extractive_answer(...)` — LLM'siz yedek
Soru kelimeleriyle chunk cümlelerini `score_text` ile puanlar, en yüksek puanlı cümleleri madde madde döner. LLM yoksa bile mantıklı, kaynağa dayalı bir cevap verir.

> **Özet zihinsel model:** RAG = "önce doğru sayfayı bul (retrieve), sonra sadece o sayfaya bakarak cevap yaz (generate), ve hangi sayfaya baktığını söyle (cite)."


In [ ]:
# Chunking mantığını LLM/embedding OLMADAN görelim (hafif, hızlı).
from src.rag_pipeline import chunk_text, load_company_documents

docs = load_company_documents()
print('Toplam doküman:', len(docs))
sample_doc = docs[0]
parts = chunk_text(sample_doc['text'], max_chars=900, overlap_chars=150)
print('Bu doküman kaç chunk oldu:', len(parts))
print('1. chunk (ilk 300 krk):')
print(parts[0][:300], '...')


c:\Users\Semih\Desktop\BizPilot AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Toplam doküman: 35
Bu doküman kaç chunk oldu: 4
1. chunk (ilk 300 krk):\n BizPilot AI product overview

BizPilot AI is an agentic RAG-powered chatbot for digital business development. It combines company-document question answering, lead qualification, personalized outreach generation, competitor-intelligence retrieval, and RAG evaluation in one Streamlit web application. ...


In [ ]:
# OPSİYONEL (ağır): Gerçek RAG retrieval. sentence-transformers modelini yükler (ilk çalıştırmada indirebilir)
# ve chroma_db/ index'inden en alakalı chunk'ları getirir. LLM GEREKTİRMEZ (sadece retrieve).
# İnternet/paket yoksa hata verebilir; o zaman bu hücreyi atla, yukarıdaki açıklamalar yeterli.
try:
    from src.rag_pipeline import retrieve, get_chroma_collection, build_index

    if get_chroma_collection(reset=False).count() == 0:
        print('Index boş, oluşturuluyor...'); build_index(reset=True)

    hits = retrieve('What is the price of the Starter plan?', top_k=3)
    for i, h in enumerate(hits, 1):
        print(f"[{i}] hybrid={h['hybrid_score']} distance={h['distance']:.3f} | {h['metadata']['title']}")
        print('    ', h['text'][:160].replace(chr(10), ' '), '...')
except Exception as exc:
    print('Retrieval çalıştırılamadı (muhtemelen paket/model yok):', type(exc).__name__, exc)


## Bölüm 11 — LLM'e gerçek çağrı: SDK'sız, sadece `urllib`

> İşlenen dosya: `src/lead_scoring_llm_explainer.py` (LLM altyapı kısmı)

Projenin bütün LLM ihtiyaçları (skor açıklaması, RAG cevabı, outreach taslağı, rakip özeti) tek bir merkezi fonksiyondan geçer: **`call_configured_llm`**. En dikkat çekici tasarım: **`openai` paketi kullanılmıyor**; istek Python'un standart `urllib` kütüphanesiyle elle atılıyor.

### Neden SDK yerine ham `urllib`?
- **Sıfır ekstra bağımlılık:** `openai` paketini kurmaya gerek yok → kurulum hafif, sürüm çakışması yok.
- **Tam kontrol:** timeout, retry, proxy — her şeyi elle yönetiyoruz.
- **Şeffaflık:** Bir HTTP POST'un içinde ne olduğunu öğrenmek için mükemmel örnek.

### `call_openai_responses(...)` — kalbi
```python
body = {
    "model": model,                       # örn. gpt-5.4
    "input": [
        {"role": "system", "content": system_prompt},   # LLM'in rolü/kuralları
        {"role": "user", "content": prompt},             # asıl istek
    ],
    "max_output_tokens": max_tokens,
}
request = urllib.request.Request(
    OPENAI_RESPONSES_URL,                 # https://api.openai.com/v1/responses
    data=json.dumps(body).encode("utf-8"),
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json", ...},
    method="POST",
)
with opener.open(request, timeout=timeout_seconds) as response:
    response_data = json.loads(response.read().decode("utf-8"))
```
Adım adım: Python sözlüğü → JSON string → bytes → HTTP POST body. `Authorization: Bearer <key>` header'ı kimlik doğrular. Cevap JSON olarak geri okunur.

### Dayanıklılık özellikleri
- **Retry döngüsü:** `for attempt in range(retry_attempts + 1)`. Sunucu geçici hata (`429, 500, 502, 503, 504`) verirse `sleep_before_retry` ile bekleyip tekrar dener. Diğer hatalarda hemen vazgeçer.
- **Timeout:** `TimeoutError`/`socket.timeout` yakalanır, gerekirse tekrar denenir.
- **Proxy desteği:** `build_url_opener` ortam değişkenlerinden (`HTTPS_PROXY` vb.) proxy okur.
- **`parse_openai_responses_response`:** Cevabın hem `output_text` kısayolunu hem de yapılandırılmış `output[].content[].text` biçimini destekler — API şekli değişse de metni bulur.

### `load_env_file()` — `.env`'i elle okumak
`python-dotenv` paketi bile gerekmiyor. `.env` dosyasını satır satır okur, `#` yorumları ve boş satırları atlar, `KEY=VALUE` çiftlerini `os.environ`'a koyar. Tırnakları temizler. Yine **sıfır bağımlılık** felsefesi.

### `clean_llm_explanation(...)` — çıktı temizliği
LLM bazen "The user wants me to..." gibi düşünce sızıntısı ya da markdown başlıkları ekler. Bu fonksiyon baştaki `#` başlıklarını ve `bad_markers` (gizli akıl yürütme kalıpları) içeren satırları temizler. Böylece kullanıcı sadece nihai, temiz cevabı görür.

> **Güvenlik notu:** API anahtarı asla koda gömülmez; `.env`'den okunur ve `.env` git'e girmez (gitignore). Anahtarların ekranda/loglarda görünmemesine dikkat edilir. (Hatırlatma: Daha önce paylaşılan gerçek OpenAI/Tavily anahtarlarını **iptal edip yenilemelisin.**)


## Bölüm 12 — Agentic Outreach üretici (çok adımlı ajan)

> İşlenen dosya: `src/outreach_agent.py`

Bu modül "agentic" kavramının somut örneği. Bir prospect (şirket + kişi + ihtiyaç) alır ve **4 adımlı bir ajan grafiği** çalıştırarak kişiselleştirilmiş bir soğuk e-posta + LinkedIn mesajı üretir:

```
qualify → research → draft → critique
```

### Neden "graph" (graf)?
Basit LLM çağrısında tek adım vardır. Burada her adım bir **node (düğüm)** ve bir *durum (state)* üzerinden ilerler. Her node state'i biraz zenginleştirip bir sonrakine devreder. Bu, karmaşık görevleri küçük, test edilebilir parçalara böler.

### `OutreachState` (TypedDict)
Ajanın hafızası. `company`, `contact`, `pain_point`, ayrıca node'ların dolduracağı `score`, `research`, `email`, `linkedin`, `steps`, `notes` alanları. `total=False` → tüm alanlar opsiyonel (node'lar zamanla doldurur).

### Node 1 — `qualify_node`: lead'i skorla
Bölüm 6'daki `score_lead`'i çağırır. Böylece mesaj, lead'in ne kadar sıcak olduğunu bilerek yazılır. Skorlama başarısız olursa `except` ile nötr skor 70'e düşer (savunmacı). **Kod yeniden kullanımı:** lead scoring modülü burada tekrar kullanılıyor.

### Node 2 — `research_node`: RAG ile zemin oluştur
Bölüm 10'daki `retrieve`'i çağırıp BizPilot'un değer önermelerini korpustan çeker (top_k=3). Böylece mesaj **gerçek ürün bilgisine** dayanır, uydurmaz. `use_rag=False` ise atlanır. Hata olursa RAG'siz devam eder (yine graceful degradation).

### Node 3 — `draft_node`: ilk taslağı yaz
LLM'e `OUTREACH_SYSTEM_PROMPT` ile (\"metrik/müşteri adı uydurma, sadece JSON döndür\") e-posta + LinkedIn taslağı yazdırır. LLM yoksa `_apply_template_draft` ile şablon taslak (yine çalışır).

### Node 4 — `critique_node`: eleştir ve iyileştir
`CRITIQUE_SYSTEM_PROMPT` ile kıdemli bir satış editörü rolünde taslağı **bir kez** rötuşlar (netlik, kısalık, kişiselleştirme). Bu, ajanın kendi çıktısını gözden geçirmesi — \"self-refinement\" deseni.

### `generate_outreach(...)` — orkestrasyon
- **LangGraph varsa:** `_build_langgraph_app` ile gerçek bir `StateGraph` kurar (node'lar kenarlarla bağlı).
- **LangGraph yoksa:** Aynı node'ları düz sırayla çağıran hafif bir runner. Sonuç aynı. Bu yüzden modül **her zaman** çalışır — LangGraph opsiyonel bir hız/görsellik katmanı, zorunluluk değil.
- `_parse_outreach_json` LLM'in döndürdüğü metinden ilk `{` ile son `}` arasını alıp JSON'a çevirir (LLM bazen fazladan metin ekler diye savunma).

> **Zihinsel model:** Bu ajan bir stajyer satış temsilcisi gibi: önce lead'i değerlendirir, sonra ürünü araştırır, sonra yazar, sonra kendi yazdığını düzeltir.


## Bölüm 13 — Rakip istihbaratı (fallback zinciri)

> İşlenen dosya: `src/competitor_intel.py`

Bu modül bir konu/rakip hakkında halka açık web bilgisi toplayıp özetler. En öğretici yanı: **üç kademeli fallback (yedekleme) zinciri.**

### `retrieve_competitor_info(...)` — kademeli deneme
```
1) Tavily API   (varsa)  → başarısız/anahtar yoksa →
2) SerpAPI      (varsa)  → başarısız/anahtar yoksa →
3) Yerel korpus (her zaman çalışır)
```
- **`search_tavily`:** Tavily'e POST atar (AI arama için tasarlanmış API).
- **`search_serpapi`:** SerpAPI'ye GET atar; sonuçları normalize eder (`link→url`, `snippet→content`) — farklı API'lerin çıktısını **tek ortak şemaya** çevirmek önemli bir desen.
- **`search_local_corpus`:** Hiç anahtar yoksa bile, `public_saas_docs.jsonl` içindeki dokümanlarda anahtar kelime skorlamasıyla arama yapar. Rakip dokümanlarına +2 bonus verir. Böylece internet olmadan bile demo çalışır.

### `run_competitor_intelligence(query, max_results, use_llm)`
Bilgiyi toplar, sonra `summarize_with_llm` (LLM varsa) veya `build_extractive_summary` (LLM yoksa, metinden çıkarımsal özet) ile özetler. Çıktı: `retrieval_provider` (hangi kaynak kullanıldı), `summary`, `results` (kaynak listesi).

> **Ders:** Gerçek sistemlerde dış servisler (API'ler) düşer, anahtarlar dolar/biter. İyi tasarım = \"birincil çalışmazsa ikincil, o da olmazsa yerel\" zinciriyle **her zaman bir cevap** üretmek.

---

## Bölüm 14 — RAG değerlendirme (RAGAS mantığı)

> İşlenen dosya: `src/rag_eval.py`

\"RAG cevabı iyi mi?\" sorusunu nasıl **ölçeriz?** Bu modül, meşhur RAGAS metriklerinin hafif, bağımsız bir versiyonunu uygular. Bir soru kümesi (`EVAL_SET`, 8 soru+referans) üzerinde üç metrik hesaplar:

### 1) Faithfulness (Sadakat)
Cevap, getirilen bağlama gerçekten dayanıyor mu, yoksa uyduruyor mu? LLM-yargıç varsa onunla, yoksa **embedding tabanlı** ölçülür: cevabın her cümlesi bağlamla ne kadar benzer (max kosinüs). Yüksek = uydurma az. **Şu an: 0.801.**

### 2) Context Precision (Bağlam İsabeti)
Getirilen chunk'ların ne kadarı gerçekten alakalı? Sıralamaya göre ağırlıklandırılır (üstteki chunk daha önemli), 0.35 eşiği kullanılır. **Şu an: 0.558.**

### 3) Answer Relevancy (Cevap Uygunluğu)
Cevap, sorulan soruyu gerçekten karşılıyor mu? Yargıç veya soru-cevap kosinüs benzerliğiyle. **Şu an: 0.732.**

### 4) Guardrail (Koruma testi)
`GUARDRAIL_SET`: kapsam dışı 2 soru. Sistem \"bu dokümanlarda yok\" diyerek **reddetmeli** (uydurmamalı). `_looks_like_refusal` ile reddedip reddetmediği kontrol edilir. **Şu an: 2/2 geçti.**

### Nasıl çalışır — `run_evaluation`
Her soru için `evaluate_item` üç metriği hesaplar, sonra ortalamalar alınır. `build_report_markdown` sonuçları `reports/week5_rag_evaluation.md`'ye yazar. `_embed` ve `_cosine` yardımcıları embedding benzerliğini hesaplar.

> **Neden önemli?** \"Çalışıyor gibi görünüyor\" mühendislik değildir. Bu modül RAG kalitesini **sayısal** hale getirir; böylece bir değişiklik (chunk boyutu, top_k, lexical_weight) metriği yükseltiyor mu düşürüyor mu görülebilir. Bu, veri bilimindeki \"ölçmeden iyileştiremezsin\" ilkesidir.


## Bölüm 15 — Streamlit arayüzü: her şeyi birleştiren kabuk

> İşlenen dosya: `app.py` (~1660 satır)

Şimdiye kadar `src/` içindeki *beyin*i inceledik. `app.py` ise *yüz*: kullanıcının tarayıcıda gördüğü 7 sekmeli arayüz. Kritik nokta — `app.py` iş mantığı içermez, sadece `src/` fonksiyonlarını **çağırır** ve sonuçları ekrana basar.

### 15.1 Streamlit nedir?
Streamlit, Python ile hızlıca web arayüzü yazmayı sağlayan bir kütüphane. HTML/JS bilmene gerek yok: `st.title(...)`, `st.button(...)`, `st.text_input(...)` gibi çağrılar otomatik bir web sayfası üretir. **Önemli davranış:** Kullanıcı bir şeye tıkladığında Streamlit **tüm scripti baştan çalıştırır.** Bu yüzden \"durum\" (state) ve \"önbellek\" (cache) mekanizmaları gerekir.

### 15.2 Kurulum başlığı
```python
st.set_page_config(...)   # sekme başlığı, ikon, geniş layout
inject_styles()           # özel CSS (bp-hero, bp-card, bp-score kartları)
```
`inject_styles` bir CSS bloğunu `st.markdown(..., unsafe_allow_html=True)` ile enjekte eder — arayüze marka görünümü verir.

### 15.3 Önbellek (cache) — neden şart?
```python
@st.cache_data
def load_dataset_preview(): ...     # CSV'yi bir kez oku, tekrar tekrar okuma

@st.cache_resource(show_spinner=False)
def ensure_rag_index(): ...         # RAG index'ini bir kez kur
```
- `@st.cache_data` → **veri** için (DataFrame, hesap sonucu). Script her çalıştığında diski tekrar okumaz.
- `@st.cache_resource` → **ağır nesneler** için (model, DB bağlantısı, index). Bir kez oluşturulur, hep aynısı kullanılır.
- Bu olmasaydı her tıklamada model yeniden yüklenir, uygulama sürünürdü.

### 15.4 Chatbot ve akıllı yönlendirme — `classify_chat_intent(text)`
Chatbot sekmesinin zekâsı burada. Kullanıcı serbest metin yazar; sistem bunu 4 kategoriye ayırıp doğru modüle yönlendirir:
```python
# anahtar kelime skorlamasıyla niyet (intent) belirle:
#   "rakip/competitor..."  -> competitor_intel
#   "email/outreach/mesaj" -> outreach_agent
#   "lead/skor/qualify"    -> lead_scoring
#   aksi halde             -> rag (doküman soru-cevap, varsayılan)
```
Bu bir mini **router**: tek sohbet kutusu, arkada dört farklı yeteneğe dağıtım. `route_chat_message` seçilen modülü çağırır, sonuç `st.session_state.chat_history`'ye eklenir (sohbet geçmişi kalıcı olsun diye).

### 15.5 `st.session_state` — durum hafızası
Script her yeniden çalıştığında sıfırlanmasın diye Streamlit bir sözlük tutar: `st.session_state`. Sohbet geçmişi, son skorlama sonucu, son rakip araması burada saklanır. Bu olmadan her tıklamada her şey silinir.

### 15.6 Yedi sekme (tab)
`main()` fonksiyonu şunları yapar: stil enjekte → dil seçici (TR/EN) → başlık → `ensure_rag_index()` → model var mı kontrol (`st.stop()` ile yoksa durur) → **7 sekme**:

| Sekme | Çağırdığı modül |
|-------|-----------------|
| Chatbot | intent router → 4 modül |
| Dashboard | veri önizleme + metrikler |
| Lead Qualification | `score_lead` + batch sonuçları |
| RAG Q&A | `answer_question` + index kur |
| Outreach Preview | `generate_outreach` |
| Competitor Intelligence | `run_competitor_intelligence` |
| Roadmap | statik yol haritası kartları |

### 15.7 `ensure_rag_index()` — ilk açılışta index kur
```python
@st.cache_resource(show_spinner=False)
def ensure_rag_index():
    count = get_chroma_collection(reset=False).count()
    if count == 0:                 # taze sunucuda chroma_db/ boşsa
        count = build_index(reset=True)   # korpustan yeniden kur
    return count
```
Neden? Çünkü sunucuya (HF Spaces / Render / kendi VPS) deploy ederken `chroma_db/` klasörü bazen gitmez. Bu fonksiyon uygulama ilk açıldığında index'i korpustan **otomatik** kurar. `@st.cache_resource` sayesinde sadece bir kez.

> **Ders — separation of concerns:** `app.py`'yi silsen bile `src/` çalışır (CLI'den). Arayüz \"ince\" tutulduğu için beyin taşınabilir, test edilebilir ve yeniden kullanılabilir. İyi mimarinin ölçüsü budur.


## Bölüm 16 — Dağıtım (deployment): laptop'tan canlı siteye

> Canlı: **https://bizpilotai.semihcankadioglu.com**

Kod bilgisayarında çalışıyor. Peki dünyanın erişebileceği bir siteye nasıl dönüşür? Bu projede kullanılan zincir:

```
Streamlit app → systemd servisi → nginx (ters vekil) → Cloudflare (DNS + HTTPS) → kullanıcı
```

### 16.1 Sunucu ve ortam
- Sunucu: bir VPS (Linux), CloudPanel paneli kurulu.
- Site kullanıcısı: `bizpilotai`, uygulama `/home/bizpilotai/app` altında.
- Python 3.12 `venv` oluşturulur, önce **CPU torch** (`--index-url .../whl/cpu`), sonra `requirements.txt` kurulur. (GPU olmadığı için CPU torch daha hafif/hızlı.)

### 16.2 `systemd` servisi — \"uygulama hep açık kalsın\"
Streamlit'i elle `streamlit run` ile başlatırsan, terminali kapatınca durur. `systemd` bunu bir **arka plan servisine** çevirir: `bizpilotai.service`. Avantajları:
- Sunucu yeniden başlasa bile uygulama otomatik açılır.
- Çökerse otomatik yeniden başlatılır.
- `127.0.0.1:8501` üzerinde dinler (sadece yerel — dışarıya nginx açar).

### 16.3 `nginx` — ters vekil (reverse proxy)
Streamlit 8501 portunda; ama kullanıcılar 80/443 (http/https) bekler. nginx gelen istekleri alıp içerideki 8501'e iletir. Streamlit **WebSocket** kullandığı için nginx yapılandırmasına özel header'lar (`Upgrade`, `Connection`) eklenir — yoksa arayüz canlı güncellenmez.

### 16.4 Cloudflare — DNS + HTTPS
Alan adı (`bizpilotai.semihcankadioglu.com`) Cloudflare üzerinden yönetilir ve **proxy'li** (turuncu bulut). Bu:
- HTTPS'i Cloudflare tarafında sağlar (\"Full\" SSL modu). Origin'de Let's Encrypt sertifikası olmasa bile site güvenli servis edilir.
- Cloudflare bir kalkan/hızlandırıcı görevi görür (CDN, DDoS koruması).

> **Deployment dersi:** \"Kodun çalışması\" ile \"kodun 7/24 canlı, güvenli ve erişilebilir olması\" farklı işlerdir. systemd (süreç yönetimi) + nginx (yönlendirme/TLS) + Cloudflare (DNS/HTTPS/CDN) klasik ve öğrenmeye değer bir üçlüdür.

> **Güvenlik hatırlatması:** Deploy sırasında `.env` içindeki gerçek OpenAI ve Tavily anahtarların ekranda göründü. Bu anahtarları **iptal edip yeniden üretmen** güvenlik açısından önemli. `.env` asla git'e girmemeli (gitignore'da kalmalı).


## Bölüm 17 — Özet, tekrar eden desenler ve alıştırmalar

### 17.1 Büyük resim — her şey nasıl bağlanıyor
```
                         ┌──────────── app.py (Streamlit, 7 sekme) ────────────┐
                         │            classify_chat_intent (router)             │
                         └───┬──────────┬───────────────┬──────────────┬───────┘
                             ▼          ▼               ▼              ▼
                    lead_scoring_   rag_pipeline   outreach_agent  competitor_intel
                    predictor.py    .py            .py             .py
                         │            │               │ (ikisini de kullanır)
                         │            │               ├── score_lead
                         │            │               └── retrieve
                         ▼            ▼
                    baseline.py   ChromaDB + embeddings
                    (model eğitir)   (46 chunk)
                         │            │
                         └── hepsi ── lead_scoring_llm_explainer.py ── OpenAI (urllib)
                                      (merkezi LLM + graceful fallback)
```
- **Tek merkezi LLM kapısı:** Her modül LLM'e `call_configured_llm` üzerinden ulaşır. Bir yerden yönetilir.
- **Kod yeniden kullanımı:** `score_lead` ve `retrieve` birden çok modülde tekrar kullanılıyor.
- **Değerlendirme ayrı:** `rag_eval.py` sistemi dışarıdan ölçer.

### 17.2 Bu projede tekrar tekrar gördüğün 6 desen (bunları aklında tut!)
1. **Graceful degradation:** API/paket yoksa çökme, daha basite düş. (LLM→şablon, Tavily→yerel, LangGraph→sıralı runner.)
2. **Separation of concerns:** `src/` beyin, `app.py` yüz. Beyin CLI'den tek başına çalışır.
3. **Sıfır gereksiz bağımlılık:** LLM çağrısı `urllib` ile, `.env` okuması elle. Az bağımlılık = az sorun.
4. **Taşınabilir yollar:** `Path(__file__).resolve().parents[1]` ile kök bulma; her yerden çalışır.
5. **Savunmacı programlama:** `try/except`, eksik değerleri `NaN`'la doldurma, çift import.
6. **Ölç, sonra iyileştir:** `rag_eval.py` metrikleri; \"çalışıyor gibi\" değil, sayıyla kanıt.

### 17.3 Kendini test et (kavram soruları)
1. RAG'de neden \"retrieve\" adımı olmadan LLM'e doğrudan sormuyoruz? (İpucu: hallucination + kaynak.)
2. `normalize_embeddings=True` neden hem index'te hem sorguda kullanılıyor?
3. Lead scoring'de `class_weight="balanced"` olmasaydı model ne yapardı?
4. `DROP_COLUMNS`'daki sütunları atmasaydık ROC-AUC neden *yanıltıcı* yükselirdi?
5. `@st.cache_resource` ile `@st.cache_data` farkı nedir?

### 17.4 Uygulamalı alıştırmalar (kodu değiştirerek öğren)
- **Kolay:** `retrieve` içindeki `lexical_weight`'i 0.25 → 0.5 yap, `rag_eval.py` çalıştır, Context Precision değişti mi?
- **Orta:** `apply_rule_adjustments`'a yeni bir kural ekle (örn. `Lead Source == "Referral" → +5`) ve batch skorların nasıl değiştiğine bak.
- **Zor:** `outreach_agent`'a 5. bir node ekle (örn. \"subject_line\" üreten). State'e alan ekle, sıralı runner'a bağla.
- **İleri:** `chunk_text`'in `max_chars`'ını 900 → 500 yap, index'i yeniden kur, retrieval isabeti nasıl etkilendi ölç.

### 17.5 Sırada ne öğrenilebilir?
- **Vektör DB'yi derinleştir:** FAISS, HNSW indeksleme, ANN algoritmaları.
- **Daha iyi RAG:** query rewriting, re-ranking modelleri (cross-encoder), çok-adımlı (multi-hop) retrieval.
- **Ajanlar:** LangGraph'ı gerçekten kurup node grafiğini görselleştir; araç (tool) çağrıları ekle.
- **MLOps:** modeli versiyonla, izle (drift), CI/CD ile otomatik deploy.

---

**Tebrikler!** Bu defteri sonuna kadar okuduysan artık BizPilot AI'ın *her* parçasının ne yaptığını ve *neden* öyle yapıldığını biliyorsun. En iyi pekiştirme: yukarıdaki alıştırmalardan en az birini gerçekten uygula ve sonucu `rag_eval.py` / batch skorlarıyla ölç.
